# Orbit Wars EDA Baseline

## Goal

Establish a **reproducible baseline** for Orbit Wars strategy work by checking the **Kaggle simulator**, running the **official starter agent**, and converting **seed-level results** into **action items** for the first non-starter bot.

## Task List

1. Detect whether the notebook is running locally or on Kaggle.
2. Confirm whether **`orbit_wars`** is available in the current runtime.
3. Prepare the **official nearest-planet starter** and a **passive fallback agent**.
4. Run a **fixed-seed starter benchmark** against `random` when the simulator is available.
5. Save **seed-level and planet-level diagnostics** for later comparison.
6. Generate a short findings report for the strategy docs.

## Artifacts Written

- `eda_environment_info.json`
- `eda_seed_summary.csv`
- `eda_planets_by_seed.csv`
- `eda_run_errors.csv`
- `eda_readme_findings.md`

Local Python currently lacks the **`orbit_wars`** environment, so local runs are dry runs only. Kaggle runs are the source of **simulation evidence**.

## 1. Runtime Setup

Detect **Kaggle versus local execution**, choose the **output folder**, and look for the **official starter bundle**. The same notebook can run locally for validation and on Kaggle for real simulator evidence.

In [ ]:
import json
import math
import platform
import shutil
import sys
import traceback
from collections import Counter, namedtuple
from pathlib import Path
from typing import Any, Optional, Sequence

import pandas as pd

CFG = {
    'n_seeds': 30,
    'primary_opponent': 'random',
    'fallback_opponent': 'passive',
    'run_simulations': True,
}

IS_KAGGLE = Path('/kaggle').exists()
WORKING = Path('/kaggle/working') if IS_KAGGLE else Path('outputs/local_eda')
WORKING.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    Path('/kaggle/input/orbit-wars'),
    Path('../data/raw/orbit-wars'),
    Path('data/raw/orbit-wars'),
]


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    """Return the first existing path from a list of candidates.

    Args:
        paths: Candidate paths ordered by preference.

    Returns:
        The first path that exists, or `None` if no candidate exists.
    """
    for path in paths:
        if path.exists():
            return path
    return None


OFFICIAL_INPUT = first_existing(INPUT_CANDIDATES)
print('is_kaggle:', IS_KAGGLE)
print('working:', WORKING)
print('official_input:', OFFICIAL_INPUT)
print('cfg:', json.dumps(CFG, indent=2))
if OFFICIAL_INPUT:
    print('official files:', sorted(p.name for p in OFFICIAL_INPUT.iterdir()))

## 2. Environment Check

Import **`kaggle_environments`**, `make`, and the Orbit Wars **`Planet` / `Fleet`** tuples. If the import fails, record the failure and skip simulation instead of treating **local dry-run output** as evidence.

In [ ]:
env_info = {
    'python': sys.version,
    'platform': platform.platform(),
    'is_kaggle': IS_KAGGLE,
    'official_input': str(OFFICIAL_INPUT) if OFFICIAL_INPUT else None,
}

ENV_AVAILABLE = False
ENV_ERROR = None
try:
    import kaggle_environments
    from kaggle_environments import make
    from kaggle_environments.envs.orbit_wars.orbit_wars import Fleet, Planet

    ENV_AVAILABLE = True
    env_info['kaggle_environments_version'] = getattr(
        kaggle_environments,
        '__version__',
        'unknown',
    )
except Exception as exc:
    ENV_ERROR = ''.join(
        traceback.format_exception_only(type(exc), exc)
    ).strip()
    Planet = namedtuple('Planet', 'id owner x y radius ships production')
    Fleet = namedtuple('Fleet', 'id owner x y angle from_planet_id ships')
    env_info['kaggle_environments_version'] = None
    env_info['environment_error'] = ENV_ERROR

env_info['orbit_wars_available'] = ENV_AVAILABLE
print(json.dumps(env_info, indent=2))

## 3. Baseline Agents

Materialize the **official nearest-planet starter** and a **passive fallback agent** in the working directory. The starter is the **control policy** that future **ROI**, **reserve**, **orbit-aware**, and **sun-safe** agents should beat.

In [ ]:
STARTER_PATH = WORKING / 'starter_main.py'
PASSIVE_PATH = WORKING / 'passive_agent.py'

if OFFICIAL_INPUT and (OFFICIAL_INPUT / 'main.py').exists():
    shutil.copyfile(OFFICIAL_INPUT / 'main.py', STARTER_PATH)
else:
    STARTER_PATH.write_text(
        '''
import math


def agent(obs):
    """Capture the nearest non-owned planet when affordable."""
    moves = []
    player = obs.get('player', 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get('planets', []) if isinstance(obs, dict) else obs.planets
    my_planets = [p for p in raw_planets if p[1] == player]
    targets = [p for p in raw_planets if p[1] != player]
    if not targets:
        return moves
    for mine in my_planets:
        nearest = min(
            targets,
            key=lambda t: math.hypot(mine[2] - t[2], mine[3] - t[3]),
        )
        ships_needed = int(nearest[5]) + 1
        if mine[5] >= ships_needed:
            angle = math.atan2(nearest[3] - mine[3], nearest[2] - mine[2])
            moves.append([int(mine[0]), float(angle), int(ships_needed)])
    return moves
'''.strip()
    )

PASSIVE_PATH.write_text(
    '''
def agent(obs):
    """Return no actions for baseline fallback simulations."""
    return []
'''.strip()
    + '\n'
)
print('starter_path:', STARTER_PATH)
print('passive_path:', PASSIVE_PATH)

## 4. Analysis Helpers

Convert raw observations into **audit-friendly tables**: **orbit classification**, final **ship-score proxy**, seed summary metrics, and planet-level map profiles.

In [ ]:
def get_field(obj: Any, name: str, default: Any = None) -> Any:
    """Read a field from either a dict-like or object-like value.

    Args:
        obj: Observation or state object from Kaggle Environments.
        name: Field name to read.
        default: Fallback value when the field is missing.

    Returns:
        Field value when present, otherwise `default`.
    """
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def rows_to_planets(rows: Optional[Sequence[Sequence[Any]]]) -> list[Any]:
    """Convert raw planet rows into Orbit Wars `Planet` tuples.

    Args:
        rows: Raw planet rows from an observation.

    Returns:
        List of `Planet`-compatible tuple objects.
    """
    return [Planet(*row) for row in rows or []]


def rows_to_fleets(rows: Optional[Sequence[Sequence[Any]]]) -> list[Any]:
    """Convert raw fleet rows into Orbit Wars `Fleet` tuples.

    Args:
        rows: Raw fleet rows from an observation.

    Returns:
        List of `Fleet`-compatible tuple objects.
    """
    return [Fleet(*row) for row in rows or []]


def is_orbiting_planet(planet: Any) -> bool:
    """Return whether a planet is inside the official rotation limit.

    Args:
        planet: `Planet`-compatible object with `x`, `y`, and `radius`.

    Returns:
        `True` when the planet rotates around the sun.
    """
    distance_to_center = math.hypot(
        float(planet.x) - 50.0,
        float(planet.y) - 50.0,
    )
    return distance_to_center + float(planet.radius) < 50.0


def score_from_obs(obs: Any, player: int) -> float:
    """Estimate final score as owned planet ships plus owned fleet ships.

    Args:
        obs: Final observation from one agent perspective.
        player: Player id to score.

    Returns:
        Ship-count score proxy for the requested player.
    """
    planets = rows_to_planets(get_field(obs, 'planets', []))
    fleets = rows_to_fleets(get_field(obs, 'fleets', []))
    planet_score = sum(
        float(p.ships) for p in planets if int(p.owner) == player
    )
    fleet_score = sum(
        float(f.ships) for f in fleets if int(f.owner) == player
    )
    return planet_score + fleet_score


def summarize_initial_planets(seed: int, obs: Any) -> list[dict[str, Any]]:
    """Create planet-level map records for a seed's initial state.

    Args:
        seed: Environment seed used for the episode.
        obs: Initial observation.

    Returns:
        Planet-level records with geometry, garrison, production, and orbit flag.
    """
    raw = get_field(obs, 'initial_planets', None) or get_field(obs, 'planets', [])
    planets = rows_to_planets(raw)
    rows = []
    for planet in planets:
        dist_center = math.hypot(float(planet.x) - 50.0, float(planet.y) - 50.0)
        rows.append({
            'seed': seed,
            'planet_id': int(planet.id),
            'owner': int(planet.owner),
            'x': float(planet.x),
            'y': float(planet.y),
            'radius': float(planet.radius),
            'ships': float(planet.ships),
            'production': float(planet.production),
            'distance_to_center': dist_center,
            'is_orbiting': is_orbiting_planet(planet),
        })
    return rows


def summarize_seed(seed: int, env: Any, opponent_label: str) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Summarize one completed environment run.

    Args:
        seed: Environment seed used for the episode.
        env: Completed Kaggle environment instance.
        opponent_label: Label for the opponent used in this run.

    Returns:
        A seed-level summary dict and planet-level initial map records.
    """
    first_state = env.steps[0][0]
    first_obs = get_field(first_state, 'observation', {})
    planet_rows = summarize_initial_planets(seed, first_obs)
    initial_rows = get_field(first_obs, 'initial_planets', None)
    planets = rows_to_planets(initial_rows or get_field(first_obs, 'planets', []))

    final_states = env.steps[-1]
    final_obs = get_field(final_states[0], 'observation', {})
    rewards = [get_field(state, 'reward', None) for state in final_states]
    statuses = [get_field(state, 'status', None) for state in final_states]
    player_scores = [
        score_from_obs(final_obs, player)
        for player in range(len(final_states))
    ]
    final_planets = rows_to_planets(get_field(final_obs, 'planets', []))
    owner_counts = Counter(int(p.owner) for p in final_planets)
    owner_production = Counter()
    for planet in final_planets:
        owner_production[int(planet.owner)] += float(planet.production)

    return {
        'seed': seed,
        'opponent': opponent_label,
        'episode_steps': len(env.steps),
        'planet_count': len(planets),
        'orbiting_planets': sum(is_orbiting_planet(p) for p in planets),
        'static_planets': sum(not is_orbiting_planet(p) for p in planets),
        'mean_initial_ships': (
            sum(float(p.ships) for p in planets) / max(len(planets), 1)
        ),
        'mean_production': (
            sum(float(p.production) for p in planets) / max(len(planets), 1)
        ),
        'max_production': max(
            [float(p.production) for p in planets],
            default=0.0,
        ),
        'rewards_json': json.dumps(rewards),
        'statuses_json': json.dumps(statuses),
        'player_scores_json': json.dumps(player_scores),
        'player0_reward': rewards[0] if rewards else None,
        'player0_score_proxy': player_scores[0] if player_scores else None,
        'player0_final_planets': owner_counts.get(0, 0),
        'player0_final_production': owner_production.get(0, 0.0),
    }, planet_rows

## 5. Seed Sweep

Run the starter over a **fixed seed set** and save the results. Future agents should use the same seeds so improvements are attributable to strategy rather than **map luck**.

In [ ]:
seed_summaries = []
planet_profiles = []
run_errors = []

N_SEEDS = int(CFG['n_seeds'])

if ENV_AVAILABLE and CFG['run_simulations']:
    for seed in range(N_SEEDS):
        try:
            env = make('orbit_wars', configuration={'seed': seed}, debug=True)
            opponent_label = str(CFG['primary_opponent'])
            try:
                env.run([str(STARTER_PATH), opponent_label])
            except Exception:
                opponent_label = str(CFG['fallback_opponent'])
                env = make('orbit_wars', configuration={'seed': seed}, debug=True)
                env.run([str(STARTER_PATH), str(PASSIVE_PATH)])
            summary, planets = summarize_seed(seed, env, opponent_label)
            seed_summaries.append(summary)
            planet_profiles.extend(planets)
            print(
                'seed',
                seed,
                'reward',
                summary['player0_reward'],
                'score_proxy',
                summary['player0_score_proxy'],
            )
        except Exception as exc:
            run_errors.append({'seed': seed, 'error': repr(exc)})
            print('seed failed', seed, repr(exc))
else:
    print('Orbit Wars environment is unavailable in this runtime. Static outputs only.')

seed_df = pd.DataFrame(seed_summaries)
planet_df = pd.DataFrame(planet_profiles)
errors_df = pd.DataFrame(run_errors)

seed_df.to_csv(WORKING / 'eda_seed_summary.csv', index=False)
planet_df.to_csv(WORKING / 'eda_planets_by_seed.csv', index=False)
errors_df.to_csv(WORKING / 'eda_run_errors.csv', index=False)

env_info['seed_count_requested'] = N_SEEDS
env_info['seed_count_completed'] = len(seed_summaries)
env_info['run_error_count'] = len(run_errors)
with open(WORKING / 'eda_environment_info.json', 'w') as f:
    json.dump(env_info, f, indent=2)

print('wrote:', WORKING / 'eda_seed_summary.csv')
print('wrote:', WORKING / 'eda_planets_by_seed.csv')
print('wrote:', WORKING / 'eda_run_errors.csv')
print('wrote:', WORKING / 'eda_environment_info.json')

## 6. Findings Summary

Write a **compact markdown report** from the generated CSVs. This bridges notebook output to **`docs/03_eda_insights.md`** and the next **strategy notebook**.

In [ ]:
lines = []
lines.append('# Orbit Wars EDA Findings')
lines.append('')
lines.append(f"- Orbit Wars environment available: `{ENV_AVAILABLE}`")
if ENV_ERROR:
    lines.append(f"- Environment import error: `{ENV_ERROR}`")
lines.append(f"- Seeds completed: `{len(seed_summaries)}` of `{N_SEEDS}`")
lines.append(f"- Run errors: `{len(run_errors)}`")

if not seed_df.empty:
    lines.append('')
    lines.append('## Seed Summary')
    lines.append('')
    lines.append(f"- Mean planet count: `{seed_df['planet_count'].mean():.2f}`")
    lines.append(f"- Mean orbiting planets: `{seed_df['orbiting_planets'].mean():.2f}`")
    lines.append(f"- Mean initial ships per planet: `{seed_df['mean_initial_ships'].mean():.2f}`")
    lines.append(f"- Mean production per planet: `{seed_df['mean_production'].mean():.2f}`")
    lines.append(f"- Mean player 0 score proxy: `{seed_df['player0_score_proxy'].mean():.2f}`")
    lines.append('')
    lines.append('## Strategy Implications To Review')
    lines.append('')
    lines.append('- Compare starter wins and losses by planet count, orbiting count, and production spread.')
    lines.append('- Check whether source-planet reserve thresholds are needed before larger attacks.')
    lines.append('- Inspect high-production orbiting targets; these likely need future-position targeting.')
    lines.append('- Add sun path checks before tuning attack aggression.')
else:
    lines.append('')
    lines.append('No simulations completed. Run this notebook on Kaggle with the Orbit Wars competition source attached.')

findings_path = WORKING / 'eda_readme_findings.md'
findings_path.write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))